# ML Project 2 - Causal Inference

## 1. Import Libraries and Load Data

In [1]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
from linearmodels.iv import IV2SLS

import warnings

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [2]:
df_desc = pd.read_stata("../data/oregonhie_descriptive_vars.dta")
df_state = pd.read_stata("../data/oregonhie_stateprograms_vars.dta")
df_ed = pd.read_stata("../data/oregonhie_ed_vars.dta")

# merge by person_id
df = df_desc.merge(df_state, on="person_id", how="left").merge(
    df_ed, on="person_id", how="left"
)

df.head()

,person_id,household_id,treatment,draw_treat,draw_lottery,applied_app,approved_app,dt_notify_lottery,dt_retro_coverage,dt_app_decision,...,ed_charg_tot_pre_ed,ed_charg_tot_ed,any_hiun_pre_ed,any_hiun_ed,num_hiun_pre_cens_ed,num_hiun_cens_ed,any_loun_pre_ed,any_loun_ed,num_loun_pre_cens_ed,num_loun_cens_ed
0,1.0,100001.0,Selected,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Submitted an Application to OHP,No,2008-08-12,2008-09-08,2008-12-31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,100002.0,Selected,Draw 6: selected in lottery 07/01/2008,Lottery Draw 6,Did NOT submit an application to OHP,No,2008-07-14,2008-08-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,100003.0,Not selected,NaN,Lottery Draw 2,NaN,NaN,2008-04-07,2008-04-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,100004.0,Not selected,NaN,Lottery Draw 8,NaN,NaN,2008-09-11,2008-10-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,100005.0,Selected,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Did NOT submit an application to OHP,No,2008-08-12,2008-09-08,NaT,...,0.0,0.0,No,No,0.0,0.0,No,No,0.0,0.0


## 2. Part(a) - Initial Data Pre-processing and Balance Check

In [3]:
# create number of people in household on lottery list (numhh_list) dummies
numhh_dummies = pd.get_dummies(
    df["numhh_list"], prefix="numhh", drop_first=True, dtype="int"
)
df = pd.concat([df, numhh_dummies], axis=1)

In [4]:
# map treatment into binary variables
df["treatment"].replace({"Selected": 1, "Not selected": 0}, inplace=True)

# handle missing value in sample_ed
df["sample_ed"] = df["sample_ed"].fillna(0)

# map female_list into binary variables
df["female_list"].replace({"1: Female": 1, "0: Male": 0}, inplace=True)

# map self_list into binary variables
df["female_list"].replace({"1: Female": 1, "0: Male": 0}, inplace=True)

df["self_list"].replace({"Signed self up": 1, "Did NOT sign self up": 0}, inplace=True)

df["any_visit_pre_ed"].replace({"Yes": 1, "No": 0}, inplace=True)

df.head()

,person_id,household_id,treatment,draw_treat,draw_lottery,applied_app,approved_app,dt_notify_lottery,dt_retro_coverage,dt_app_decision,...,any_hiun_pre_ed,any_hiun_ed,num_hiun_pre_cens_ed,num_hiun_cens_ed,any_loun_pre_ed,any_loun_ed,num_loun_pre_cens_ed,num_loun_cens_ed,numhh_signed self up + 1 additional person,numhh_signed self up + 2 additional people
0,1.0,100001.0,1,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Submitted an Application to OHP,No,2008-08-12,2008-09-08,2008-12-31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,2.0,100002.0,1,Draw 6: selected in lottery 07/01/2008,Lottery Draw 6,Did NOT submit an application to OHP,No,2008-07-14,2008-08-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,3.0,100003.0,0,NaN,Lottery Draw 2,NaN,NaN,2008-04-07,2008-04-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,4.0,100004.0,0,NaN,Lottery Draw 8,NaN,NaN,2008-09-11,2008-10-08,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,5.0,100005.0,1,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Did NOT submit an application to OHP,No,2008-08-12,2008-09-08,NaT,...,No,No,0.0,0.0,No,No,0.0,0.0,0,0


### Full sample balance check

In [5]:
y = df["sample_ed"]

X = pd.concat([df[["treatment"]], numhh_dummies], axis=1)
X = sm.add_constant(X)

full_model = sm.OLS(endog=y, exog=X, missing="drop")
result = full_model.fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:              sample_ed   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     59.03
Date:                Mon, 02 Mar 2026   Prob (F-statistic):           4.17e-38
Time:                        20:11:15   Log-Likelihood:                -49627.
No. Observations:               74922   AIC:                         9.926e+04
Df Residuals:                   74918   BIC:                         9.930e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

### Sample balance checks


In [6]:
# restrict to ED samples
df_ed_sample = df[df["sample_ed"] != 0.0].copy()

df_ed_sample.shape

(24646, 146)

In [7]:
X_sample = pd.concat(
    [df_ed_sample[["treatment"]], df_ed_sample[numhh_dummies.columns]], axis=1
)
X_sample = sm.add_constant(X_sample)

X_sample

,const,treatment,numhh_signed self up + 1 additional person,numhh_signed self up + 2 additional people
4,1.0,1,0,0
7,1.0,0,1,0
8,1.0,0,0,0
15,1.0,0,1,0
17,1.0,0,0,0
...,...,...,...,...
74906,1.0,1,0,0
74910,1.0,1,1,0
74914,1.0,1,0,0
74917,1.0,0,0,0


#### (i) Year of birth

In [8]:
balance_yob = sm.OLS(endog=df_ed_sample["birthyear_list"], exog=X_sample, missing="drop")
result_balance_yob = balance_yob.fit()
print(result_balance_yob.summary())

                            OLS Regression Results                            
Dep. Variable:         birthyear_list   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.825
Date:                Mon, 02 Mar 2026   Prob (F-statistic):              0.140
Time:                        20:11:15   Log-Likelihood:                -96304.
No. Observations:               24646   AIC:                         1.926e+05
Df Residuals:                   24642   BIC:                         1.926e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

#### (ii) Female

In [9]:
balance_female = sm.OLS(endog=df_ed_sample["female_list"], exog=X_sample, missing="drop")
result_balance_female = balance_female.fit()
print(result_balance_female.summary())

                            OLS Regression Results                            
Dep. Variable:            female_list   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     15.03
Date:                Mon, 02 Mar 2026   Prob (F-statistic):           9.01e-10
Time:                        20:11:15   Log-Likelihood:                -17757.
No. Observations:               24646   AIC:                         3.552e+04
Df Residuals:                   24642   BIC:                         3.556e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

#### (iii) Signed up self for lottery

In [10]:
balance_self = sm.OLS(endog=df_ed_sample["self_list"], exog=X_sample, missing="drop")
result_balance_self = balance_self.fit()
print(result_balance_self.summary())

                            OLS Regression Results                            
Dep. Variable:              self_list   R-squared:                       0.446
Model:                            OLS   Adj. R-squared:                  0.446
Method:                 Least Squares   F-statistic:                     6611.
Date:                Mon, 02 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:11:15   Log-Likelihood:                 1774.3
No. Observations:               24646   AIC:                            -3541.
Df Residuals:                   24642   BIC:                            -3508.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

#### (iv) Any ED visit, pre-randomization (censored)

In [11]:
balance_visit_pre_ed = sm.OLS(endog=df_ed_sample["any_visit_pre_ed"], exog=X_sample, missing="drop")
result_balance_visit_pre_ed = balance_visit_pre_ed.fit()
print(result_balance_visit_pre_ed.summary())

                            OLS Regression Results                            
Dep. Variable:       any_visit_pre_ed   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                     124.2
Date:                Mon, 02 Mar 2026   Prob (F-statistic):           7.55e-80
Time:                        20:11:16   Log-Likelihood:                -15848.
No. Observations:               24646   AIC:                         3.170e+04
Df Residuals:                   24642   BIC:                         3.174e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

#### (v) Number of ED visits, pre-randomization (censored)

In [12]:
balance_num_visit = sm.OLS(endog=df_ed_sample["num_visit_pre_cens_ed"], exog=X_sample, missing="drop")
result_balance_num_visit = balance_num_visit.fit()
print(result_balance_num_visit.summary())

                              OLS Regression Results                             
Dep. Variable:     num_visit_pre_cens_ed   R-squared:                       0.010
Model:                               OLS   Adj. R-squared:                  0.009
Method:                    Least Squares   F-statistic:                     78.89
Date:                   Mon, 02 Mar 2026   Prob (F-statistic):           8.76e-51
Time:                           20:11:16   Log-Likelihood:                -50165.
No. Observations:                  24634   AIC:                         1.003e+05
Df Residuals:                      24630   BIC:                         1.004e+05
Df Model:                              3                                         
Covariance Type:               nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

## 3. Part(b) - Causal Effect of Being Selected by Lottery

In [13]:
# restrict to Portland ED sample
df_b = df_ed_sample.copy()

In [14]:
# rename for easier reference
df_b.rename(columns={"ohp_all_ever_matchn_30sep2009": "medicaid_enroll"}, inplace=True)

# map to binary variables
df_b["medicaid_enroll"].replace({"Enrolled": 1, "NOT enrolled": 0}, inplace=True)

df_b.head()

,person_id,household_id,treatment,draw_treat,draw_lottery,applied_app,approved_app,dt_notify_lottery,dt_retro_coverage,dt_app_decision,...,any_hiun_pre_ed,any_hiun_ed,num_hiun_pre_cens_ed,num_hiun_cens_ed,any_loun_pre_ed,any_loun_ed,num_loun_pre_cens_ed,num_loun_cens_ed,numhh_signed self up + 1 additional person,numhh_signed self up + 2 additional people
4,5.0,100005.0,1,Draw 7: selected in lottery 08/01/2008,Lottery Draw 7,Did NOT submit an application to OHP,No,2008-08-12,2008-09-08,NaT,...,No,No,0.0,0.0,No,No,0.0,0.0,0,0
7,8.0,102094.0,0,NaN,Lottery Draw 8,NaN,NaN,2008-09-11,2008-10-08,NaT,...,No,Yes,0.0,2.0,No,No,0.0,0.0,1,0
8,9.0,100009.0,0,NaN,Lottery Draw 1,NaN,NaN,2008-03-10,2008-03-11,NaT,...,Yes,No,1.0,0.0,No,No,0.0,0.0,0,0
15,16.0,140688.0,0,NaN,Lottery Draw 2,NaN,NaN,2008-04-07,2008-04-08,NaT,...,Yes,Yes,1.0,5.0,No,No,0.0,0.0,1,0
17,18.0,100018.0,0,NaN,Lottery Draw 4,NaN,NaN,2008-05-09,2008-06-09,NaT,...,No,No,0.0,0.0,Yes,No,2.0,0.0,0,0


In [15]:
features = ["treatment", "birthyear_list", "female_list", "self_list", "any_visit_pre_ed", "num_visit_pre_cens_ed"]

features = pd.concat(
    [df_ed_sample[features], df_ed_sample[numhh_dummies.columns]], axis=1
)
features = sm.add_constant(features)

medicaid_enroll_causal = sm.OLS(endog=df_b["medicaid_enroll"], exog=features, missing="drop")
result_medicaid_enroll_causal = medicaid_enroll_causal.fit()
print(result_medicaid_enroll_causal.summary())

                            OLS Regression Results                            
Dep. Variable:        medicaid_enroll   R-squared:                       0.098
Model:                            OLS   Adj. R-squared:                  0.098
Method:                 Least Squares   F-statistic:                     334.2
Date:                Mon, 02 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:11:16   Log-Likelihood:                -12745.
No. Observations:               24634   AIC:                         2.551e+04
Df Residuals:                   24625   BIC:                         2.558e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

## 4. Part(c) - Causal Effect of Enrolling into a Medicaid Program on ED Visits

In [16]:
df_c = df_b.copy()

y_any = df_c["any_visit_ed"]
y_num = df_c["num_visit_cens_ed"]

### First Stage

In [17]:
first_stage = sm.OLS(endog=df_c["medicaid_enroll"],exog=X_sample, missing="drop")

result_first_stage = first_stage.fit()
print(result_first_stage.summary())

                            OLS Regression Results                            
Dep. Variable:        medicaid_enroll   R-squared:                       0.077
Model:                            OLS   Adj. R-squared:                  0.077
Method:                 Least Squares   F-statistic:                     687.5
Date:                Mon, 02 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:11:16   Log-Likelihood:                -13032.
No. Observations:               24646   AIC:                         2.607e+04
Df Residuals:                   24642   BIC:                         2.611e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

### Second Stage

#### (i) Outcome: Any ED visit (binary)

In [18]:
features_iv = ["birthyear_list", "female_list", "self_list", "any_visit_pre_ed", "num_visit_pre_cens_ed"]

features_iv = pd.concat(
    [df_ed_sample[features_iv], df_ed_sample[numhh_dummies.columns]], axis=1
)
features_iv = sm.add_constant(features_iv)

iv_any = IV2SLS(dependent=df_c["any_visit_ed"], endog=df_c["medicaid_enroll"], exog=features_iv, instruments=df_c["treatment"])
res_iv_any_2sls = iv_any.fit()
print(res_iv_any_2sls)

                          IV-2SLS Estimation Summary                          
Dep. Variable:       any_visit_ed.Yes   R-squared:                      0.1722
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1719
No. Observations:               24634   F-statistic:                    5751.9
Date:                Mon, Mar 02 2026   P-value (F-stat)                0.0000
Time:                        20:11:16   Distribution:                  chi2(8)
Cov. Estimator:                robust                                         
                                                                              
                                             Parameter Estimates                                              
                                            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------------------------------
const                              

#### (ii) Outcome: Number of ED visits (censored)

In [19]:
iv_num = IV2SLS(dependent=df_c["num_visit_cens_ed"], endog=df_c["medicaid_enroll"], exog=features_iv, instruments=df_c["treatment"])
res_iv_num_2sls = iv_num.fit()
print(res_iv_num_2sls)

                          IV-2SLS Estimation Summary                          
Dep. Variable:      num_visit_cens_ed   R-squared:                      0.3398
Estimator:                    IV-2SLS   Adj. R-squared:                 0.3396
No. Observations:               24615   F-statistic:                    2684.8
Date:                Mon, Mar 02 2026   P-value (F-stat)                0.0000
Time:                        20:11:16   Distribution:                  chi2(8)
Cov. Estimator:                robust                                         
                                                                              
                                             Parameter Estimates                                              
                                            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------------------------------
const                              